In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS tfl_pipeline")
spark.sql("CREATE SCHEMA IF NOT EXISTS tfl_pipeline.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS tfl_pipeline.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS tfl_pipeline.gold")

In [0]:
import sys
import json
import requests
import time
from datetime import datetime, timezone

sys.path.append("/Workspace/Users/ichibakry7@gmail.com/tfl-tracker/src")
from tfl_client import (
    fetch_all_lines,
    fetch_arrivals,
    fetch_disruptions,
    fetch_stop_points,
    fetch_timetable
)
from config import (
    MODES,
    BRONZE_ARRIVALS,
    BRONZE_DISRUPTIONS,
    BRONZE_STOP_POINTS,
    BRONZE_TIMETABLES
)

#Config
APP_KEY = dbutils.secrets.get(scope="tfl_scope", key="tfl_api_key")
INGESTED_AT = datetime.now(timezone.utc).isoformat()

def save_to_bronze(data_list, table_name):
    """
    A helper to save data consistently. 
    It converts to a DataFrame, removes duplicates, and saves.
    """
    if not data_list:
        print(f"No data to save for {table_name}")
        return

    df = spark.createDataFrame(data_list)
    
    # Simple deduplication: if the exact same payload exists in this batch, drop it
    deduped_df = df.dropDuplicates()
    
    (deduped_df.write
        .format("delta")
        .mode("append") 
        .option("mergeSchema", "true") # Useful if the API adds new fields later
        .saveAsTable(table_name))
    
    print(f"Saved {deduped_df.count()} records to {table_name}")

start_time = time.time() # track how long it takes to run

# Get all lines
try:
    print("Discovering lines...")
    lines = fetch_all_lines(APP_KEY, MODES)
    print(f"Found {len(lines)} lines. Starting ingestion...")
except Exception as e:
    print(f"CRITICAL: Could not fetch lines. Error: {e}")
    lines = []

#Process Arrivals & Disruptions
arrival_records = []
disruption_records = []

for i, line in enumerate(lines, 1):
    print(f"  [{i}/{len(lines)}] Fetching arrivals & disruptions for: {line}...")

    try:
        # Arrivals
        arrivals = fetch_arrivals(APP_KEY, line)
        for a in arrivals:
            arrival_records.append({
                "line_id": line,
                "raw_payload": json.dumps(a),
                "ingested_at": INGESTED_AT
            })
        
        # Disruptions
        disruptions = fetch_disruptions(APP_KEY, line)
        for d in disruptions:
            disruption_records.append({
                "line_id": line,
                "raw_payload": json.dumps(d),
                "ingested_at": INGESTED_AT
            })
    except Exception as e:
        print(f"Error fetching data for line {line}: {e}")

save_to_bronze(arrival_records, BRONZE_ARRIVALS)
save_to_bronze(disruption_records, BRONZE_DISRUPTIONS)

#Process Stop Points & Timetables
stop_records = []
timetable_records = []
seen_naptan = set()

for i, line in enumerate(lines, 1):
    print(f"  [{i}/{len(lines)}] Working on: {line}...", end="")
    try:
        stops = fetch_stop_points(APP_KEY, line)
        print(f" Done! (Found {len(stops)} stops)")

        for s in stops:
            naptan_id = s.get("naptanId")
            station_name = s.get("commonName", "Unknown Station")
            
            # Save unique stops
            if naptan_id and naptan_id not in seen_naptan:
                seen_naptan.add(naptan_id)
                stop_records.append({
                    "naptan_id": naptan_id,
                    "raw_payload": json.dumps(s),
                    "ingested_at": INGESTED_AT
                })
            
            # Fetch timetable for this stop
            try:
                timetable = fetch_timetable(APP_KEY, line, naptan_id)
                timetable_records.append({
                    "line_id": line,
                    "naptan_id": naptan_id,
                    "raw_payload": json.dumps(timetable),
                    "ingested_at": INGESTED_AT
                })
            except requests.exceptions.HTTPError as e:
                if e.response.status_code == 404:
                    print(f"No timetable exists for {line} at {station_name} ({naptan_id})")
                else:
                    print(f"Unexpected error for {line} at {station_name}: {e}")
            except Exception as e:
                print(f"General error: {e}")
                
    except Exception as e:
        print(f"Error processing stops for line {line}: {e}")

save_to_bronze(stop_records, BRONZE_STOP_POINTS)
save_to_bronze(timetable_records, BRONZE_TIMETABLES)

total_seconds = time.time() - start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print(f"Ingestion Finished!")
print(f"Total Duration: {minutes}m {seconds}s")

